# Structure visualization
Visual structures in the databases. Get a feel for diversity.

#TODO expand with different methods (maybe build on what has been done in 9_svr.ipynb)

In [ ]:
import sys
from pathlib import Path
import random

notebookdir = Path.cwd().parent
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw

import sqlalchemy as sa
from sqlalchemy.orm import sessionmaker
from src.db_schema import *
from src.db_utils import get_basic_data
from src.log_utils import log_section, log_to_file

import ipywidgets as widgets
from IPython.display import display


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)

# Paths and DB setup
SCRIPT_DIR = notebookdir
DATA_DIR = SCRIPT_DIR / "processed_data"
DATABASE_FILE_HSBD = DATA_DIR / "hsbd_t_half_data.db"
ENGINE_HSBD = sa.create_engine(f"sqlite:///{DATABASE_FILE_HSBD}")
Session_HSBD = sessionmaker(bind=ENGINE_HSBD)

DATABASE_FILE_VEGA = DATA_DIR / "vega_t_half_data.db"
ENGINE_VEGA = sa.create_engine(f"sqlite:///{DATABASE_FILE_VEGA}")
Session_VEGA = sessionmaker(bind=ENGINE_VEGA)


# # Logging setup
# log_time = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
# logs_dir = SCRIPT_DIR / "logs" / "visualization" / log_time
# logs_dir.mkdir(parents=True, exist_ok=True)
# log_file = str(logs_dir / "vis.log")

# log_to_file("Visualization", log_file)
# log_to_file(f"Notebook run started. Working dir: {SCRIPT_DIR}", log_file)
# print(f"Log file: {log_file}")

In [ ]:
# Load data
data_air = get_basic_data("air", Session_HSBD)
data_water = get_basic_data("water", Session_HSBD)
data_soil = get_basic_data("soil", Session_HSBD)
data_sediment = get_basic_data("sediment", Session_HSBD)

# not air data in VEGA, only water, soil and sediment
data_water_vega = get_basic_data("water", Session_VEGA)
data_soil_vega = get_basic_data("soil", Session_VEGA)
data_sediment_vega = get_basic_data("sediment", Session_VEGA)

In [ ]:
def draw_compound_structures(df, n=20, random=False):
    """Draws the first n compound structures from the DataFrame. if random bool, then random n compounds are drawn."""

    df = df["Canonical_smiles"]
    if random:
        smiles_list = df.sample(n, random_state=42).tolist()
    else:
        # Get the first n SMILES strings
        smiles_list = df.head(n).tolist()

    mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]
    img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(350, 200))
    display(img)

In [ ]:
# static draw of first 20 compounds of a compartment
draw_compound_structures(data_air, n=20, random=True)

In [ ]:
smiles_data = {}
for name, df in [
    ("air_hsbd", data_air),
    ("water_hsbd", data_water),
    ("soil_hsbd", data_soil),
    ("sediment_hsbd", data_sediment),
    ("water_vega", data_water_vega),
    ("soil_vega", data_soil_vega),
    ("sediment_vega", data_sediment_vega),
]:
    smiles_data[name] = df["Canonical_smiles"].tolist()

In [ ]:
# Interactive viewer for train/test compound structures
N_DISPLAY = 20

# Dispose prior viewer so rerunning this cell keeps a single live panel.
_prev = globals().get("_svr_viewer")
if _prev is not None:
    _prev["dropdown"].unobserve(_prev["on_change"], names="value")
    _prev["refresh_btn"].on_click(_prev["on_refresh"], remove=True)
    _prev["dropdown"].close()
    _prev["refresh_btn"].close()
    _prev["info_label"].close()
    _prev["ui"].close()

dropdown = widgets.Dropdown(
    options=list(smiles_data.keys()),
    value=list(smiles_data.keys())[0],
    description="Set:",
    layout=widgets.Layout(width="180px"),
)
refresh_btn = widgets.Button(
    description="Refresh",
    button_style="info",
    layout=widgets.Layout(width="100px"),
)
info_label = widgets.Label()

viewer_state = {"image_handle": None}


def render(_=None):
    dataset = dropdown.value
    smiles_list = smiles_data[dataset]
    n = len(smiles_list)

    if n > N_DISPLAY:
        chosen = random.sample(smiles_list, N_DISPLAY)
        info_label.value = f"{dataset}: {n:,} compounds - showing {N_DISPLAY} random"
    else:
        chosen = smiles_list
        info_label.value = f"{dataset}: {n:,} compounds - showing all"

    mols = [Chem.MolFromSmiles(s) for s in chosen]
    img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(350, 200))

    if viewer_state["image_handle"] is None:
        viewer_state["image_handle"] = display(img, display_id=True)
    else:
        viewer_state["image_handle"].update(img)


def on_change(change):
    render(change["new"])


def on_refresh(_):
    render()


ui = widgets.HBox([dropdown, refresh_btn, info_label])
display(ui)
render()

dropdown.observe(on_change, names="value")
refresh_btn.on_click(on_refresh)

globals()["_svr_viewer"] = {
    "dropdown": dropdown,
    "refresh_btn": refresh_btn,
    "info_label": info_label,
    "ui": ui,
    "on_change": on_change,
    "on_refresh": on_refresh,
}

In [ ]:
# count compounds containing elements in each dataset
def count_element_compounds(smiles_list, element):
    count = sum(1 for e in smiles_list if element in e)
    return count


elements = ["S", "N", "Cl", "Br", "F"]
datasets = list(smiles_data.keys())

# results = []
# for dataset in datasets:
#     for elem in elements:
#         count = count_element_compounds(smiles_data[dataset], elem)
#         results.append({"Dataset": dataset, "Element": elem, "count": count})

# element_counts_df = pd.DataFrame(results)
# display(element_counts_df)


compartments = ["air", "water", "soil", "sediment"]
side_by_side = []
for comp in compartments:
    hsbd_key = f"{comp}_hsbd"
    vega_key = f"{comp}_vega"
    for elem in elements:
        row = {
            "Element": elem,
            "Dataset_HSBD": hsbd_key,
            "Dataset_Size_HSBD": len(smiles_data[hsbd_key]) if hsbd_key in smiles_data else 0,
            "count_HSBD": int(count_element_compounds(smiles_data[hsbd_key], elem)) if hsbd_key in smiles_data else 0,
        }
        if vega_key in smiles_data:
            row["Dataset_VEGA"] = vega_key
            row["Dataset_Size_VEGA"] = len(smiles_data[vega_key])
            row["count_VEGA"] = int(count_element_compounds(smiles_data[vega_key], elem))
        else:
            row["Dataset_VEGA"] = "-"
            row["Dataset_Size_VEGA"] = "-"
            row["count_VEGA"] = "-"
        side_by_side.append(row)

element_counts_pivot_df = pd.DataFrame(side_by_side)

display(element_counts_pivot_df)